# 03 Generate Images Occ

**Purpose.** Occupancy-masked / TIC-normalised image generation (the improved variant).

**Thesis reference.** §4.5.1 Image Representations

**Inputs / outputs.** See the data-layout table in the repository README. Paths near the top of the notebook (e.g. `PROFILE_DIR`, `IMAGE_ROOT`, `LABELS`) are relative to the repository root and assume the folder layout described there.

---

In [1]:
import os, json, time
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image

DATA_ROOT = "../ahen_data/filtered_tsv_2"
LABELS    = "labels/labels_severity_all.json"
IMG       = 224
RT_RANGE  = (0.0, 37.0)
MZ_RANGE  = (50.0, 1650.0)
MZ_FINE   = 8000                                   # fine grid for occupancy scan
OCC_MASK  = True        # keep only m/z bins ever populated
TIC_NORM  = True        # normalise each patient by total ion current
MZ_BANDS  = 3           # RGB: number of m/z bands (equal-occupancy if masked)
OUT       = Path(f"images_v2{'_occ' if OCC_MASK else ''}{'_tic' if TIC_NORM else ''}")

for sub in ("grayscale_heatmap", "rgb_mz_split"):
    (OUT/sub).mkdir(parents=True, exist_ok=True)

lab = json.load(open(LABELS))["labels"]

def pid_of(fn):
    return os.path.splitext(os.path.basename(fn))[0].replace("WCmsdata_s","").lstrip("0") or "0"

def norm_pid(raw):
    d = "".join(c for c in raw if c.isdigit())
    return f"P{int(d):04d}" if d else None

files = []
for f in sorted(Path(DATA_ROOT).glob("*.tsv")):
    p = norm_pid(pid_of(f.name))
    if p in lab: files.append((f, p))
print(f"{len(files)} files match the label set")

def read_full(path):
    """Stream the whole file, return mz, time, intensity as arrays."""
    mz, tm, it = [], [], []
    for chunk in pd.read_csv(path, sep="\t", names=["time","mz","intensity"],
                             dtype=str, low_memory=False, chunksize=2_000_000):
        for c in ("time","mz","intensity"):
            chunk[c] = pd.to_numeric(chunk[c], errors="coerce")
        chunk = chunk.dropna()
        tm.append(chunk["time"].values); mz.append(chunk["mz"].values)
        it.append(chunk["intensity"].values)
    return (np.concatenate(mz), np.concatenate(tm), np.concatenate(it))

# pass 1: occupancy across the cohort
if OCC_MASK:
    print("pass 1: scanning occupied m/z regions")
    occ = np.zeros(MZ_FINE, dtype=np.int32)
    t0 = time.time()
    for k,(f,p) in enumerate(files,1):
        mz,_,it = read_full(f)
        m = (mz>=MZ_RANGE[0])&(mz<MZ_RANGE[1])&(it>0)
        idx = ((mz[m]-MZ_RANGE[0])/(MZ_RANGE[1]-MZ_RANGE[0])*MZ_FINE).astype(int)
        occ[np.clip(idx,0,MZ_FINE-1)] += 1
        if k%50==0: print(f"  {k}/{len(files)}  {(time.time()-t0)/60:.0f} min")
    # keep bins populated in at least 5% of patients
    keep = occ >= 0.05*len(files)
    edges = np.linspace(MZ_RANGE[0], MZ_RANGE[1], MZ_FINE+1)
    kept_mz = edges[:-1][keep]
    print(f"  occupied fraction of m/z axis: {keep.mean():.1%}")
    np.save(OUT/"occupied_mz.npy", kept_mz)
    # build a lookup that maps a real m/z to a compacted pixel column
    order = np.argsort(kept_mz)
    def mz_to_col(v):
        # position within the kept masses, scaled to IMG
        idx = np.searchsorted(kept_mz, v)
        return np.clip((idx/len(kept_mz)*IMG).astype(int), 0, IMG-1)
else:
    def mz_to_col(v):
        return np.clip(((v-MZ_RANGE[0])/(MZ_RANGE[1]-MZ_RANGE[0])*IMG).astype(int),0,IMG-1)

def rt_to_row(v):
    return np.clip(((v-RT_RANGE[0])/(RT_RANGE[1]-RT_RANGE[0])*IMG).astype(int),0,IMG-1)

# pass 2: write images
print("\npass 2: writing images")
t0 = time.time()
for k,(f,p) in enumerate(files,1):
    mz,tm,it = read_full(f)
    m = (mz>=MZ_RANGE[0])&(mz<MZ_RANGE[1])&(tm>=RT_RANGE[0])&(tm<RT_RANGE[1])&(it>0)
    mz,tm,it = mz[m],tm[m],it[m]
    if TIC_NORM and it.sum()>0: it = it/it.sum()*1e6

    col = mz_to_col(mz); row = rt_to_row(tm)

    # greyscale
    H = np.zeros((IMG,IMG),dtype=np.float64)
    np.add.at(H,(row,col),it)
    H = np.log1p(H); H = H/ (H.max() or 1)
    Image.fromarray((H*255).astype(np.uint8),mode="L").save(
        OUT/"grayscale_heatmap"/f"{p}.png")

    # RGB by m/z band (equal-occupancy bands if masked)
    if OCC_MASK:
        bands = np.array_split(np.sort(kept_mz), MZ_BANDS)
        cuts = [(b[0],b[-1]) for b in bands]
    else:
        e = np.linspace(*MZ_RANGE, MZ_BANDS+1); cuts=list(zip(e[:-1],e[1:]))
    chans=[]
    for lo,hi in cuts:
        s=(mz>=lo)&(mz<=hi)
        Hc=np.zeros((IMG,IMG)); np.add.at(Hc,(row[s],col[s]),it[s])
        Hc=np.log1p(Hc); Hc=Hc/(Hc.max() or 1)
        chans.append((Hc*255).astype(np.uint8))
    Image.fromarray(np.stack(chans,-1),mode="RGB").save(
        OUT/"rgb_mz_split"/f"{p}.png")
    if k%50==0: print(f"  {k}/{len(files)}  {(time.time()-t0)/60:.0f} min")

print(f"\ndone in {(time.time()-t0)/60:.0f} min")
print(f"images in {OUT}/")

510 files match the label set
pass 1: scanning occupied m/z regions
  50/510  33 min
  100/510  67 min
  200/510  138 min
  250/510  174 min
  300/510  211 min
  350/510  246 min
  400/510  281 min
  450/510  314 min
  500/510  349 min
  occupied fraction of m/z axis: 100.0%

pass 2: writing images
  50/510  35 min
  100/510  72 min
  150/510  107 min
  200/510  143 min
  250/510  180 min
  300/510  218 min
  350/510  253 min
  400/510  289 min
  450/510  323 min
  500/510  356 min

done in 364 min
images in images_v2_occ_tic/


In [ ]:
print(a)